# Architecture Ablation

Runs the architecture sweep with optional reaction-force loss and optional Hessian regularization. Edit the config cell before running.

In [ ]:
import os
import numpy as np

from properties import SlinkyN3Properties
from run_architectures import (
    SweepConfig,
    run_architecture_sweep,
    subset_energy_only,
    subset_main_paper_candidates,
    subset_all,
    plot_summary_final_losses,
)


In [ ]:
# Dataset / physical setup
properties = SlinkyN3Properties(mass=0.3)
train_file = "../simulation_data_2D/3_noded/n3_slinky_sim_train_dataset.npz"
valid_file = "../simulation_data_2D/3_noded/n3_slinky_sim_test_dataset.npz"

# Pick one subset, or pass None to run all registered architectures.
# selected_architectures = subset_energy_only()
# selected_architectures = subset_main_paper_candidates()
selected_architectures = subset_all()
# selected_architectures = None

# Optional force loss. Leave strength at 0.0 for displacement-only training.
force_loss_strength = 1
force_key = None          # None auto-detects F/forces/reaction_force when force loss is enabled
force_components = (0,)   # e.g. (0,) for Fx, or (0, 1, 2)
force_sign = 1.0
return_loss_components = force_loss_strength != 0.0

# Optional Hessian regularizer. Leave strength at 0.0 to disable.
hessian_reg_strength = 0.0
hessian_reg_probes = 1
hessian_reg_seed = 0


In [ ]:
cfg = SweepConfig(
    output_dir="arch_ablation_outputs_n3_slinky_simdata",
    n_epochs=500,
    lr=1e-2,
    seed=42,
    hidden=(10,),
    input_mode="invariant",
    activation="tanh",
    corr_factor=0.01,
    only_stretching_NN=False,
    zero_reference=True,
    valid_every=1,
    max_dlambda=5e-2,
    iters=20,
    ls_steps=10,
    abs_tol=1e-4,
    rel_tol=1e-4,
    train_fail_on_nonconvergence=False,
    prediction_fail_on_nonconvergence=False,
    hessian_reg_strength=hessian_reg_strength,
    hessian_reg_probes=hessian_reg_probes,
    hessian_reg_seed=hessian_reg_seed,
    force_key=force_key,
    force_loss_strength=force_loss_strength,
    force_components=force_components,
    force_sign=force_sign,
    return_loss_components=return_loss_components,
    save_npz=True,
    save_model=True,
    save_plots=True,
    save_energy_landscapes=True,
    verbose=True,
)


In [ ]:
results = run_architecture_sweep(
    properties=properties,
    train_file=train_file,
    valid_file=valid_file,
    cfg=cfg,
    selected_architectures=selected_architectures,
)


In [ ]:
summary_plot = os.path.join(cfg.output_dir, "final_loss_summary.png")
plot_summary_final_losses(results, save_path=summary_plot, show=False)
summary_plot
